# MDQ Tutorial: Molecular Dynamics DockQ

This notebook shows how to use **MDQ** step by step.

You will:

1. Load a trajectory.
2. Calculate the interface metrics (FSR, iRMSD, lRMSD) and the MDQ score.
3. Cluster the frames (DBSCAN and hierarchical).
4. Export the representative structures and the cluster tables.

> Run the cells one by one, from top to bottom.
> For more details, read the MDQ user guide (`MDQ.md`).

## 0. Before you start

You need:

- `mdq.py` in the same folder as this notebook;
- a trajectory file (for example `production.xtc`);
- a topology file (for example `topology.pdb`);
- the packages listed in the user guide (`mdtraj`, `numpy`, `pandas`, `scikit-learn`, `scipy`, `umap-learn`, `matplotlib`, `seaborn`, `tqdm`).

Install command (run it in a terminal, not in the notebook):

```bash
conda install -c conda-forge python mdtraj numpy pandas tqdm matplotlib seaborn scikit-learn scikit-learn-extra scipy umap-learn jupyter ipykernel
```

## 1. Import the classes

In [ ]:
from mdq import TrajectoryAnalyzer, TrajectoryClusterVisualizer

## 2. Set your parameters

Change the values in this cell for your own system.

- `chains` uses **MDTraj residue indices** (they start at 0). They are **not** the PDB residue numbers.
- `stride=10` loads one frame out of 10. A larger stride is faster. Use `stride=1` for the final analysis.
- The number of calculations grows with the **square** of the number of frames. Start with a small number of frames to test.

In [ ]:
traj_file = "production.xtc"
top_file = "topology.pdb"

chains = {
    "A": range(0, 314),
    "B": range(314, 634),
}

stride = 10

## 3. Create the `TrajectoryAnalyzer`

This loads the trajectory and finds the backbone atoms (`N`, `CA`, `C`, `O`) of each chain.

In [ ]:
analyzer = TrajectoryAnalyzer(
    traj_file=traj_file,
    top_file=top_file,
    chains=chains,
    stride=stride,
)

print("Number of loaded frames:", analyzer.traj.n_frames)

## 4. Calculate the metrics

Run these four methods **in this order**.

| Method | What it calculates |
|--------|--------------------|
| `compute_shared_fraction()` | FSR: fraction of shared residue–residue contacts |
| `compute_irmsd()` | Interface RMSD (weighted and unweighted) |
| `compute_lrmsd()` | Ligand RMSD (weighted and unweighted). It needs `compute_irmsd()` first |
| `compute_mdq()` | MDQ score, from the three metrics above |

This step can take a long time for many frames.

In [ ]:
analyzer.compute_shared_fraction(min_A=2, max_A=5)   # contacts between 2 and 5 Å
analyzer.compute_irmsd()
analyzer.compute_lrmsd()
analyzer.compute_mdq()

### Look at the results

In [ ]:
analyzer.df_score.head()

The most important columns are:

- `shared_fraction`: FSR (0 to 1);
- `irmsd_weighted` and `lrmsd_weighted`: RMSD values in Å;
- `mdq_weighted`: the MDQ score (0 to 1). A higher value means more similar frames.

| MDQ | Meaning |
|-----|---------|
| > 0.80 | Very high similarity |
| 0.60 – 0.80 | High similarity |
| 0.40 – 0.60 | Moderate similarity |
| 0.20 – 0.40 | Low similarity |
| < 0.20 | Very low similarity |

## 5. Save the MDQ table

`compute_mdq()` does **not** save `mdq.csv` by itself. Save it now. The visualizer needs this file.

In [ ]:
analyzer.df_score.to_csv("mdq.csv", index=False)

## 6. Create the `TrajectoryClusterVisualizer`

The visualizer reads `mdq.csv` and creates `distance.csv`, with:

```text
distance = 1 - mdq_weighted
```

A small distance means that two frames are similar.

In [ ]:
visualizer = TrajectoryClusterVisualizer()

## 7. Look at the distance matrix

- **Heatmap:** the distance between all pairs of frames. Red cells are missing values.
- **Clustermap:** the same matrix, but the frames are reordered to show natural groups.

These plots help you to choose the clustering parameters.

In [ ]:
visualizer.plot_distance_heatmap()

In [ ]:
visualizer.plot_distance_clustermap()

## 8. DBSCAN clustering

Important parameters:

- `eps_dict`: the distance threshold for each interface. A **smaller** `eps` is **stricter**.
- `min_samples_pct`: the fraction of frames used for DBSCAN `min_samples`.
- `min_pct_dyn`: the minimum cluster size (fraction). Smaller clusters become noise (`-1`).
- `total_frames_dyn`: the total number of frames, used to calculate the cluster size.

The plots show MDS, UMAP and t-SNE projections. Colours are clusters, grey points are noise, and the circled points are the representative frames.

In [ ]:
n_frames = analyzer.traj.n_frames

visualizer.run_dbscan_analysis(
    eps_dict={"A-B": 0.20},
    min_samples_pct=0.05,
    min_pct_dyn=0.05,
    total_frames_dyn=n_frames,
)

## 9. Hierarchical clustering

Important parameters:

- `method`: the linkage method (`"single"`, `"complete"`, `"average"` or `"ward"`).
- `threshold`: the distance used to cut the dendrogram.
- `min_pct` and `total_frames`: the minimum cluster size, like for DBSCAN.

You do not need to choose the number of clusters.

In [ ]:
visualizer.run_hierarchical_analysis(
    method="average",
    threshold=0.3,
    min_pct=0.05,
    total_frames=n_frames,
)

## 10. Get the frames of each cluster

For each interface and each cluster, you get the list of frames and the **representative frame**. The representative frame is the frame with the smallest mean distance to the other frames of the cluster.

In [ ]:
dbscan_table, hierarchical_table = visualizer.get_cluster_frames_table()

print("DBSCAN:")
print(dbscan_table)
print()
print("Hierarchical:")
print(hierarchical_table)

## 11. Export the results

### Representative structures (PDB)

> Use the **same stride** as in the analysis. If not, the exported frames will not match the analysed frames.
> The call below follows the user guide. If your version of `mdq.py` has a `stride` argument for this method, pass `stride=stride` too.

In [ ]:
visualizer.export_representative_frames_pdb(
    traj_file=traj_file,
    top_file=top_file,
    output_dir="representative_structures",
    cluster_tables=(dbscan_table, hierarchical_table),
)

### Cluster table and statistics (CSV)

In [ ]:
visualizer.export_cluster_representatives_csv(output_file="cluster_representatives.csv")
visualizer.save_cluster_stats_csv(output_prefix="cluster_stats")

## 12. Files created

| File | Content |
|------|---------|
| `shared_fraction.csv` | FSR for all frame pairs |
| `mdq.csv` | All metrics and MDQ scores |
| `distance.csv` | Distance table (`1 - mdq_weighted`) |
| `representative_structures/` | One PDB file per representative frame |
| `cluster_representatives.csv` | Frames and representative frame of each cluster |
| `cluster_stats_DBSCAN.csv` | Statistics of the DBSCAN clusters |
| `cluster_stats_Hierarchical.csv` | Statistics of the hierarchical clusters |

## Common problems

- **`SyntaxError: non-default argument follows default argument`:** fix the argument order in `run_dbscan_analysis()` and `run_hierarchical_analysis()` in `mdq.py`. Arguments without a default value must come first.
- **`FileNotFoundError: mdq.csv`:** run the cell that saves `mdq.csv` (step 5).
- **Wrong chains:** check that `chains` uses zero-based MDTraj residue indices.
- **Unexpected number of clusters:** look at the heatmap and the dendrogram, then change `eps` (DBSCAN) or `threshold` (hierarchical).
- **Too slow:** increase `stride` to use fewer frames.

## To report your results

Write down the stride, the number of frames, `min_A` and `max_A`, the chain definitions, the clustering method and all its parameters.